# Pipeline IoT - Spark Structured Streaming

## Arquitectura Kappa: Procesamiento en tiempo real de datos ambientales IoT

Este notebook consume eventos de Kafka, aplica ventanas temporales con watermarking, calcula agregaciones y escribe los resultados en TimescaleDB.

## 1. Configuracion de Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("IoT-Streaming-Pipeline") \
    .config("spark.sql.streaming.checkpointLocation", "/home/jovyan/checkpoint/iot") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session creada - Version:", spark.version)

Spark Session creada - Version: 3.5.0


## 2. Definicion del Schema del evento

In [2]:
iot_schema = StructType([
    StructField("estacion", StringType(), True),
    StructField("temperatura", DoubleType(), True),
    StructField("humedad", DoubleType(), True),
    StructField("presion", DoubleType(), True),
    StructField("altura", DoubleType(), True),
    StructField("gas", DoubleType(), True),
    StructField("iaq", DoubleType(), True),
    StructField("eco2", DoubleType(), True),
    StructField("VOC", DoubleType(), True),
    StructField("calidad_aire", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("event_timestamp", LongType(), True),
    StructField("delayed", BooleanType(), True),
])

print("Schema definido:")
iot_schema.printTreeString()

Schema definido:


AttributeError: 'StructType' object has no attribute 'printTreeString'

## 3. Lectura del stream de Kafka

In [3]:
KAFKA_BROKER = "kafka:29092"
TOPIC = "iot.air_quality.streaming"

raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", TOPIC) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("Stream de Kafka configurado correctamente")

Stream de Kafka configurado correctamente


## 4. Parseo y transformacion de datos

In [4]:
parsed_stream = raw_stream.select(
    from_json(col("value").cast("string"), iot_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select(
    "data.*",
    "kafka_timestamp",
    (col("kafka_timestamp").cast("long") - col("event_timestamp")).alias("latencia_ms")
).withColumn(
    "event_time",
    to_timestamp(col("created_at"))
)

print("Stream parseado y latencia calculada")

Stream parseado y latencia calculada


## 5. Ventanas, Watermarking y Agregaciones

In [5]:
WINDOW_DURATION = "1 minute"
SLIDE_DURATION = "30 seconds"
WATERMARK_DURATION = "2 minutes"

windowed_stream = parsed_stream \
    .withWatermark("event_time", WATERMARK_DURATION) \
    .groupBy(
        window("event_time", WINDOW_DURATION, SLIDE_DURATION),
        "estacion"
    ) \
    .agg(
        round(avg("temperatura"), 2).alias("avg_temperatura"),
        round(avg("humedad"), 2).alias("avg_humedad"),
        round(avg("iaq"), 2).alias("avg_iaq"),
        round(avg("presion"), 2).alias("avg_presion"),
        round(avg("eco2"), 2).alias("avg_eco2"),
        round(avg("VOC"), 3).alias("avg_voc"),
        count("*").alias("evento_count"),
        round(avg("latencia_ms"), 2).alias("latencia_ms")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "estacion",
        "avg_temperatura",
        "avg_humedad",
        "avg_iaq",
        "avg_presion",
        "avg_eco2",
        "avg_voc",
        "evento_count",
        "latencia_ms"
    )

print(f"Ventanas: {WINDOW_DURATION} con slide de {SLIDE_DURATION}")
print(f"Watermark: {WATERMARK_DURATION}")

Ventanas: 1 minute con slide de 30 seconds
Watermark: 2 minutes


## 6. Escritura en TimescaleDB

In [6]:
TSDB_URL = "jdbc:postgresql://timescaledb:5432/iot_metrics"
TSDB_PROPS = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

query = windowed_stream.writeStream \
    .outputMode("update") \
    .format("jdbc") \
    .option("url", TSDB_URL) \
    .option("dbtable", "air_quality_metrics") \
    .option("user", TSDB_PROPS["user"]) \
    .option("password", TSDB_PROPS["password"]) \
    .option("driver", TSDB_PROPS["driver"]) \
    .option("checkpointLocation", "/home/jovyan/checkpoint/iot") \
    .trigger(processingTime="10 seconds") \
    .start()

print("Query de escritura en TimescaleDB iniciado")

UnsupportedOperationException: Data source jdbc does not support streamed writing.

## 7. Monitor en consola (debug)

In [7]:
console_query = windowed_stream.writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", "false") \
    .option("numRows", 5) \
    .trigger(processingTime="10 seconds") \
    .start()

print("Monitor de consola iniciado")

Monitor de consola iniciado


## 8. Estado del streaming

In [8]:
import time

print("Estado de las queries activas:")
for q in spark.streams.active:
    print(f"  - {q.name} | ID: {q.id} | Activa: {q.isActive}")

print("\nEsperando datos... (detener con query.stop())")

Estado de las queries activas:

Esperando datos... (detener con query.stop())


## 9. Detener queries

In [ ]:
for q in spark.streams.active:
    q.stop()
    print(f"Query {q.id} detenida")

spark.stop()